In [103]:
import pandas as pd

df = pd.read_csv("diabetes.csv")
df.isnull().sum()

Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64

In [104]:
cols_with_zeros = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
df[cols_with_zeros] = df[cols_with_zeros].replace(0, pd.NA)
df.fillna(df.median(), inplace=True)

C:\Users\Kartik\AppData\Local\Temp\ipykernel_11508\2952167308.py:3: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df.fillna(df.median(), inplace=True)


In [105]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X = df.drop("Outcome", axis = 1)
Y = df["Outcome"]

In [106]:
X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size= 0.2, random_state= 42
)

In [107]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [108]:
import torch
import torch.nn as nn

X_train_tensor = torch.tensor(X_train_scaled, dtype = torch.float32)
Y_train_tensor = torch.tensor(Y_train.values, dtype = torch.float32).view(-1,1)

X_test_tensor = torch.tensor(X_test_scaled, dtype = torch.float32)
Y_test_tensor = torch.tensor(Y_test.values, dtype = torch.float32).view(-1,1)

In [109]:
from torch.utils.data import DataLoader, TensorDataset

train_dataset = TensorDataset(X_train_tensor, Y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, Y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size = 32, shuffle = True)
test_loader = DataLoader(test_dataset, batch_size = 32)

In [127]:
# creating ANN
class DiabetesNet(nn.Module):
    def __init__(self):
        super().__init__()

        self.model = nn.Sequential(
            nn.Linear(X_train.shape[1],32),
            nn.ReLU(),
            nn.Dropout(0.2),
        
            nn.Linear(32,16),
            nn.ReLU(),

            nn.Linear(16,1),
            nn.Sigmoid()
        )
    def forward(self,x):
        return self.model(x)


In [128]:
#optimizer

import torch.optim as optim

model = DiabetesNet()

criteria = nn.BCELoss()
optimizer = optim.Adam(model.parameters())

In [129]:
epochs = 200

for epoch in range(epochs):
    model.train()
    for x_batch, y_batch in train_loader:
        optimizer.zero_grad()

        output = model(x_batch)
        loss = criteria(output, y_batch)
        loss.backward()

        optimizer.step()
        
    if(epoch+1) % 10 ==0:
        print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

Epoch 10, Loss: 0.2594
Epoch 20, Loss: 0.4663
Epoch 30, Loss: 0.3273
Epoch 40, Loss: 1.7092
Epoch 50, Loss: 0.2655
Epoch 60, Loss: 0.2893
Epoch 70, Loss: 0.5057
Epoch 80, Loss: 0.2505
Epoch 90, Loss: 0.5136
Epoch 100, Loss: 0.3429
Epoch 110, Loss: 0.3590
Epoch 120, Loss: 0.1085
Epoch 130, Loss: 0.1478
Epoch 140, Loss: 0.2231
Epoch 150, Loss: 0.1417
Epoch 160, Loss: 0.3381
Epoch 170, Loss: 0.2466
Epoch 180, Loss: 0.3361
Epoch 190, Loss: 0.5564
Epoch 200, Loss: 0.1884


In [131]:
sets = {"Train": train_dataset, 
        "Test":  Y_train_tensor}

model.eval()
with torch.no_grad():
    for name, (X, y) in sets.items():
        acc = ((model(X) > 0.5).float() == y).float().mean()
        print(f"{name} Accuracy: {acc.item()*100:.2f}%")

ValueError: too many values to unpack (expected 2)